In [2]:
import pandas as pd
import numpy as np
import glob
import helper_functions
import helper_functions
from sklearn.model_selection import train_test_split
import pandas as pd
from xgboost import XGBRegressor
import xgboost as xgb
import tensorflow as tf
mape = tf.keras.losses.MeanAbsolutePercentageError()
mse = tf.keras.losses.MeanSquaredError()
mae = tf.keras.losses.MeanAbsoluteError()
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RepeatedKFold
from sklearn.metrics import make_scorer
from sklearn.metrics import mean_absolute_percentage_error
import numpy as np
import sklearn
import pandas as pd

2023-12-06 12:14:13.503338: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


# Get AWS Data

In [3]:
# lets load in the AWS data
data_dir = '../app_data_cloud/aws-data-summer-2023/'


aws_data = pd.DataFrame()
for file_name in glob.glob(data_dir+'*.csv'):
    x = pd.read_csv(file_name, low_memory=False)
    aws_data = pd.concat([aws_data,x],axis=0)

usable_data = aws_data[aws_data['REALTIME (sec)'].isna()==False]
aws_realtime_data = usable_data[['machine', 'args', 'ranks', 'REALTIME (sec)']]

# Prep AWS Data

In [12]:
lc_data = helper_functions.get_data()
lc_data = lc_data[lc_data['REALTIME (sec)'].isna()==False]
lc_data = lc_data.rename(columns={'machine': 'source machine'})

In [15]:
def get_aws_pairs(aws_data: pd.DataFrame, lc_data:pd.DataFrame) -> pd.DataFrame:
    new_data_df: pd.DataFrame = None
    all_args = aws_data.args.unique()
    for args in all_args:
        aws_filtered_by_args = aws_data[aws_data.args == args]
        lc_filtered_by_args = lc_data[lc_data.args == args]
        all_ranks = aws_data.ranks.unique()
        for ranks in all_ranks:
            aws_filtered_by_ranks_and_args = aws_filtered_by_args[aws_filtered_by_args.ranks == ranks]
            lc_filtered_by_ranks_and_args = lc_filtered_by_args[lc_filtered_by_args.ranks == ranks]
            all_aws_machines = aws_filtered_by_ranks_and_args['machine'].unique()
            all_lc_machines = lc_filtered_by_ranks_and_args['source machine'].unique()
            for lc_machine in all_lc_machines:
                # print('num all machines', len(all_machines))
                temp_lc_df = lc_filtered_by_ranks_and_args[lc_filtered_by_ranks_and_args['source machine'] == lc_machine].copy(deep=True)
                base_lc_df = temp_lc_df.copy(deep=True)
                for i in range (1, len(all_aws_machines)):
                    temp_lc_df = pd.concat([temp_lc_df, base_lc_df.copy(deep=True)])
                temp_lc_df['target machine'] = all_aws_machines.copy()
                new_data_df = pd.concat([new_data_df, temp_lc_df])
    return new_data_df
aws_pairs = get_aws_pairs(aws_realtime_data, lc_data)

In [19]:
def calc_aws_relative_performance(aws_pairs: pd.DataFrame, aws_data) -> pd.DataFrame:
    new_df = aws_pairs.copy(deep=True)
    for args in aws_pairs.args.unique():
        filtered_by_args = aws_pairs[aws_pairs.args == args]
        for ranks in filtered_by_args.ranks.unique():
            filtered_by_ranks_and_args = filtered_by_args[filtered_by_args.ranks == ranks]
            for machine in filtered_by_ranks_and_args['source machine'].unique():
                filtered_by_machine = filtered_by_ranks_and_args[filtered_by_ranks_and_args['source machine'] == machine]
                for target_machine in filtered_by_machine['target machine'].unique():
                    target_time = aws_data[(aws_data['args'] == args) & (aws_data['ranks'] == ranks) & (aws_data['machine'] == target_machine)]['REALTIME (sec)'].values[0]
                    source_time = aws_pairs[(aws_pairs['args'] == args) & (aws_pairs['ranks'] == ranks) & (aws_pairs['source machine'] == machine) & (aws_pairs['target machine'] == target_machine)]['REALTIME (sec)'].values[0]
                    rel_perf = target_time / source_time
                    new_df.loc[filtered_by_machine.index, 'relative performance'] = rel_perf
    return new_df
aws_all = calc_aws_relative_performance(aws_pairs, aws_realtime_data)
aws_all = helper_functions.merge_benchmarks(aws_all)

/home/alex/Programming/Research/LLNL/alex-performance-modeling/medium-data-models/helper_functions.py:117: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  raja_df.machine[raja_df['machine'] == 'ec2-c5n'] = 'ec2-c5.metal'
/home/alex/Programming/Research/LLNL/alex-performance-modeling/medium-data-models/helper_functions.py:118: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  raja_df.machine[raja_df['machine'] == 'ec2-c6i'] = 'ec2-c6i.metal'


# Get and Prep LC Data

In [20]:
lc_all = helper_functions.get_data()
lc_all = helper_functions.drop_unmatched_rows(lc_all)
lc_all = lc_all.rename(columns={'machine': 'source machine'})
lc_all = helper_functions.create_machine_combinations(lc_all)
lc_all = helper_functions.merge_benchmarks(lc_all, False)
lc_all = helper_functions.calc_relative_performances(lc_all)


/home/alex/Programming/Research/LLNL/alex-performance-modeling/medium-data-models/helper_functions.py:117: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  raja_df.machine[raja_df['machine'] == 'ec2-c5n'] = 'ec2-c5.metal'
/home/alex/Programming/Research/LLNL/alex-performance-modeling/medium-data-models/helper_functions.py:118: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  raja_df.machine[raja_df['machine'] == 'ec2-c6i'] = 'ec2-c6i.metal'


# Train on LC, Test on AWS

In [21]:
refined_lc_df = helper_functions.remove_unneeded_columns(lc_all)
refined_aws_df = helper_functions.remove_unneeded_columns(aws_all)
run_data_lc, relative_performance_values_lc = helper_functions.split_x_y(refined_lc_df)
run_data_aws, relative_performance_values_aws = helper_functions.split_x_y(refined_aws_df)

In [22]:
n_estimators = 1500
max_depth = 6

In [24]:
model = XGBRegressor(n_estimators=n_estimators, max_depth=max_depth)
model.fit(run_data_lc, relative_performance_values_lc)
y_pred = model.predict(run_data_aws)
aws_test_mape = mean_absolute_percentage_error(relative_performance_values_aws, y_pred)
print('MAPE', aws_test_mape)

/home/alex/Programming/Tools/miniconda3/envs/ml/lib/python3.9/site-packages/xgboost/data.py:299: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(dtype):
/home/alex/Programming/Tools/miniconda3/envs/ml/lib/python3.9/site-packages/xgboost/data.py:301: FutureWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, CategoricalDtype) instead
  elif is_categorical_dtype(dtype) and enable_categorical:
/home/alex/Programming/Tools/miniconda3/envs/ml/lib/python3.9/site-packages/xgboost/data.py:332: FutureWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, CategoricalDtype) instead
  if is_categorical_dtype(dtype)
/home/alex/Programming/Tools/miniconda3/envs/ml/lib/python3.9/site-packages/xgboost/data.py:323: FutureWarning: is_categorical_dtype is deprecated and will be removed in a future versio

MAPE 1994.4497573980086


/home/alex/Programming/Tools/miniconda3/envs/ml/lib/python3.9/site-packages/xgboost/data.py:299: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(dtype):
/home/alex/Programming/Tools/miniconda3/envs/ml/lib/python3.9/site-packages/xgboost/data.py:301: FutureWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, CategoricalDtype) instead
  elif is_categorical_dtype(dtype) and enable_categorical:
/home/alex/Programming/Tools/miniconda3/envs/ml/lib/python3.9/site-packages/xgboost/data.py:332: FutureWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, CategoricalDtype) instead
  if is_categorical_dtype(dtype)
/home/alex/Programming/Tools/miniconda3/envs/ml/lib/python3.9/site-packages/xgboost/data.py:323: FutureWarning: is_categorical_dtype is deprecated and will be removed in a future versio

# Train and Test with LC, AWS Mix

In [25]:
# x_train, x_test, y_train, y_test
lc_x_train, lc_x_test, lc_y_train, lc_y_test = train_test_split(run_data_lc, relative_performance_values_lc, test_size=0.2, random_state=42)
aws_x_train, aws_x_test, aws_y_train, aws_y_test = train_test_split(run_data_aws, relative_performance_values_aws, test_size=0.2, random_state=42)

x_train = pd.concat([lc_x_train, aws_x_train])
y_train = pd.concat([lc_y_train, aws_y_train])
x_test = pd.concat([lc_x_test, aws_x_test])
y_test = pd.concat([lc_y_test, aws_y_test])

In [26]:
model = XGBRegressor(n_estimators=n_estimators, max_depth=max_depth)
model.fit(x_train, y_train)
y_pred_mixed = model.predict(x_test)
y_pred_aws = model.predict(aws_x_test)
mixed_mape = mean_absolute_percentage_error(y_test, y_pred_mixed)
aws_mape = mean_absolute_percentage_error(aws_y_test, y_pred_aws)
print('Mixed MAPE', mixed_mape)
print('AWS MAPE', aws_mape)

/home/alex/Programming/Tools/miniconda3/envs/ml/lib/python3.9/site-packages/xgboost/data.py:299: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(dtype):
/home/alex/Programming/Tools/miniconda3/envs/ml/lib/python3.9/site-packages/xgboost/data.py:301: FutureWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, CategoricalDtype) instead
  elif is_categorical_dtype(dtype) and enable_categorical:
/home/alex/Programming/Tools/miniconda3/envs/ml/lib/python3.9/site-packages/xgboost/data.py:332: FutureWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, CategoricalDtype) instead
  if is_categorical_dtype(dtype)
/home/alex/Programming/Tools/miniconda3/envs/ml/lib/python3.9/site-packages/xgboost/data.py:323: FutureWarning: is_categorical_dtype is deprecated and will be removed in a future versio

Mixed MAPE 101.25755459257704
AWS MAPE 1145.6128765438198


/home/alex/Programming/Tools/miniconda3/envs/ml/lib/python3.9/site-packages/xgboost/data.py:299: FutureWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(dtype):
/home/alex/Programming/Tools/miniconda3/envs/ml/lib/python3.9/site-packages/xgboost/data.py:301: FutureWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, CategoricalDtype) instead
  elif is_categorical_dtype(dtype) and enable_categorical:
/home/alex/Programming/Tools/miniconda3/envs/ml/lib/python3.9/site-packages/xgboost/data.py:332: FutureWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, CategoricalDtype) instead
  if is_categorical_dtype(dtype)
/home/alex/Programming/Tools/miniconda3/envs/ml/lib/python3.9/site-packages/xgboost/data.py:323: FutureWarning: is_categorical_dtype is deprecated and will be removed in a future versio